In [1]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\marti\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import requests
import time

df = pd.read_csv("person_names.csv")

In [ ]:
import pandas as pd
import requests
import time

df = pd.read_csv("person_names.csv")

# Headers necesarios para que Wikidata acepte las solicitudes
HEADERS = {
    'User-Agent': 'DataMiningProject/1.0 (Educational Purpose)',
    'Accept': 'application/json'
}

def clean_name(name):
    """Limpia el nombre eliminando ' & family' y otros sufijos"""
    name = name.replace(" & family", "")
    name = name.strip()
    return name

def is_business_person(entity_data, entity_id):
    """Verifica si la entidad es un empresario/billonario"""
    claims = entity_data["entities"][entity_id].get("claims", {})
    
    # P106 = occupation (ocupación)
    if "P106" in claims:
        for claim in claims["P106"]:
            try:
                occupation_id = claim["mainsnak"]["datavalue"]["value"]["id"]
                # IDs relevantes: Q131524=entrepreneur, Q43845=businessperson, Q140686=chairperson
                # Q484876=CEO, Q1273818=investor, Q205375=philanthropist
                if occupation_id in ["Q131524", "Q43845", "Q140686", "Q484876", "Q1273818", "Q205375"]:
                    return True
            except (KeyError, TypeError):
                continue
    
    # Verificar descripción
    descriptions = entity_data["entities"][entity_id].get("descriptions", {})
    desc_en = descriptions.get("en", {}).get("value", "").lower()
    business_keywords = ["business", "entrepreneur", "billionaire", "investor", "ceo", "founder"]
    if any(keyword in desc_en for keyword in business_keywords):
        return True
    
    return False

def get_company_count(name, max_retries=3):
    """Obtiene el número de compañías asociadas a una persona desde Wikidata"""
    
    # Limpiar el nombre
    clean = clean_name(name)
    
    for attempt in range(max_retries):
        try:
            # Paso 1: Buscar la entidad de la persona (buscar más resultados)
            search_url = "https://www.wikidata.org/w/api.php"
            search_params = {
                "action": "wbsearchentities",
                "search": clean,
                "language": "en",
                "format": "json",
                "type": "item",
                "limit": 5  # Buscar hasta 5 resultados
            }
            
            search_response = requests.get(
                search_url, 
                params=search_params, 
                headers=HEADERS,
                timeout=30
            )
            search_response.raise_for_status()
            search_data = search_response.json()
            
            if not search_data.get("search"):
                print(f"  ⚠️ No se encontró entidad para: {clean}")
                return 0
            
            # Intentar encontrar la persona correcta (empresario/billonario)
            entity_id = None
            label = None
            
            for result in search_data["search"]:
                candidate_id = result["id"]
                candidate_label = result.get('label', '')
                candidate_desc = result.get('description', '').lower()
                
                # Verificar si la descripción sugiere que es un empresario
                business_keywords = ["business", "entrepreneur", "billionaire", "investor", "ceo", "founder", "chairperson"]
                if any(keyword in candidate_desc for keyword in business_keywords):
                    entity_id = candidate_id
                    label = candidate_label
                    print(f"  ✓ Encontrado: {entity_id} - {label} ({candidate_desc})")
                    break
            
            # Si no encontramos uno obvio, usar el primero
            if not entity_id:
                entity_id = search_data["search"][0]["id"]
                label = search_data["search"][0].get('label', '')
                print(f"  ⚠️ Usando primer resultado: {entity_id} - {label}")
            
            # Paso 2: Obtener datos de la entidad
            entity_url = f"https://www.wikidata.org/wiki/Special:EntityData/{entity_id}.json"
            entity_response = requests.get(
                entity_url, 
                headers=HEADERS,
                timeout=30
            )
            entity_response.raise_for_status()
            entity_data = entity_response.json()
            
            claims = entity_data["entities"][entity_id].get("claims", {})
            
            # Verificar si es empresario (ayuda a confirmar identidad)
            if is_business_person(entity_data, entity_id):
                print(f"  ✓ Confirmado como empresario/inversor")
            
            # Propiedades de Wikidata relacionadas con compañías (expandidas)
            company_properties = {
                "P112": "founder of",           # fundador de
                "P169": "chief executive officer", # CEO de
                "P488": "chairperson",          # presidente de
                "P1037": "director/manager",    # director/gerente de
                "P108": "employer",             # empleador
                "P1830": "owner of",            # propietario/inversor de
                "P127": "owned by",             # propiedad de (inverso, para verificar)
                "P1454": "legal form",          # forma legal (para holdings)
                "P749": "parent organization",  # organización matriz
                "P102": "member of political party"  # a veces relevante para magnates
            }
            
            companies = set()
            property_counts = {}
            
            # Recolectar todas las compañías/organizaciones
            for prop_id, prop_name in company_properties.items():
                if prop_id in claims:
                    prop_count = 0
                    for claim in claims[prop_id]:
                        try:
                            # Obtener el ID de la compañía
                            company_id = claim["mainsnak"]["datavalue"]["value"]["id"]
                            companies.add(company_id)
                            prop_count += 1
                        except (KeyError, TypeError):
                            continue
                    if prop_count > 0:
                        property_counts[prop_name] = prop_count
            
            count = len(companies)
            
            # Logging detallado
            if count > 0:
                print(f"  → Compañías encontradas: {count}")
                if property_counts:
                    print(f"     Desglose: {', '.join([f'{k}: {v}' for k, v in property_counts.items()])}")
            else:
                print(f"  ⚠️ 0 compañías - revisar manualmente")
            
            return count
            
        except requests.exceptions.Timeout:
            if attempt < max_retries - 1:
                print(f"  ⏱ Timeout, reintentando ({attempt + 1}/{max_retries})...")
                time.sleep(2)
                continue
            else:
                print(f"  ✗ Timeout después de {max_retries} intentos")
                return 0
                
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                print(f"  ⚠️ Error: {str(e)[:50]}, reintentando...")
                time.sleep(2)
                continue
            else:
                print(f"  ✗ Error de conexión: {str(e)[:100]}")
                return 0
                
        except Exception as e:
            print(f"  ✗ Error: {str(e)[:100]}")
            return 0
    
    return 0


# Verificar conexión primero
print("Verificando conexión a Wikidata...")
try:
    test_response = requests.get(
        "https://www.wikidata.org/w/api.php?action=wbsearchentities&search=test&language=en&format=json",
        headers=HEADERS,
        timeout=15
    )
    if test_response.status_code == 200:
        print("✓ Conexión exitosa\n")
    else:
        print(f"⚠️ Código de respuesta: {test_response.status_code}\n")
except Exception as e:
    print(f"✗ Error de conexión: {e}")
    print("Verifica tu conexión a internet o proxy\n")

# Procesar cada nombre
results = []

print("Buscando compañías para cada persona...\n")

for idx, name in enumerate(df["personName"], 1):
    print(f"{idx}. {name}")
    
    count = get_company_count(name)
    results.append(count)
    
    # Pausa para no sobrecargar la API
    time.sleep(1.5)
    print()

# Guardar resultados
df["numberOfCompanies"] = results

df[["personName", "numberOfCompanies"]].to_csv(
    "person_companies.csv",
    index=False
)

print("✓ Resultados guardados en 'person_companies.csv'")
print(f"\nResumen:")
print(df[["personName", "numberOfCompanies"]])

Verificando conexión a Wikidata...
✓ Conexión exitosa

Buscando compañías para cada persona...

1. Bernard Arnault & family
  ✓ Encontrado: Q32055 - Bernard Arnault
  → Compañías encontradas: 0

2. Elon Musk
  ✓ Encontrado: Q317521 - Elon Musk
  → Compañías encontradas: 14

3. Jeff Bezos
  ✓ Encontrado: Q312556 - Jeff Bezos
  → Compañías encontradas: 6

4. Larry Ellison
  ✓ Encontrado: Q92759 - Larry Ellison
  → Compañías encontradas: 9

5. Warren Buffett
  ✓ Encontrado: Q47213 - Warren Buffett
  → Compañías encontradas: 1

6. Bill Gates
  ✓ Encontrado: Q5284 - Bill Gates
  → Compañías encontradas: 6

7. Michael Bloomberg
  ✓ Encontrado: Q607 - Michael Bloomberg
  → Compañías encontradas: 2

8. Carlos Slim Helu & family
  ✓ Encontrado: Q170419 - Carlos Slim
  → Compañías encontradas: 6

9. Mukesh Ambani
  ✓ Encontrado: Q298547 - Mukesh Ambani
  → Compañías encontradas: 5

10. Steve Ballmer
  ✓ Encontrado: Q181162 - Steve Ballmer
  → Compañías encontradas: 2

11. Francoise Bettencourt M